## **MLlib: 영화 추천 알고리즘 구현**
+ 데이터셋: MovieLens(2500만개 영화 평점 데이터), 본 예제에서는 7만개의 데이터만 사용 ```ratings_short.csv```
+ 추천 알고리즘: Alternating Least Squares (ALS)

---

### **1. 영화 평점 데이터 불러오기 & 데이터프레임 생성하기**

In [4]:
# [+] SparkSession 설정
from pyspark.sql import SparkSession
spark=SparkSession.builder.master('local').appName('movie_recommendation').getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 04:15:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### movielens 데이터 불러오기
+ 파일명: ```ratings_short.csv```
+ 스키마 설정: ```inferSchema=True```
+ 헤더 사용: ```header=True```

In [5]:
# movielens 데이터 불러오기
path = '../data/'
file = 'ratings_short.csv'

ratings_df = spark.read.csv(
    path + file, 
    inferSchema=True, 
    header=True
)

In [6]:
# 데이터프레임 출력
ratings_df.show()

+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|     1|    296|   5.0|1147880044|
|     1|    306|   3.5|1147868817|
|     1|    307|   5.0|1147868828|
|     1|    665|   5.0|1147878820|
|     1|    899|   3.5|1147868510|
|     1|   1088|   4.0|1147868495|
|     1|   1175|   3.5|1147868826|
|     1|   1217|   3.5|1147878326|
|     1|   1237|   5.0|1147868839|
|     1|   1250|   4.0|1147868414|
|     1|   1260|   3.5|1147877857|
|     1|   1653|   4.0|1147868097|
|     1|   2011|   2.5|1147868079|
|     1|   2012|   2.5|1147868068|
|     1|   2068|   2.5|1147869044|
|     1|   2161|   3.5|1147868609|
|     1|   2351|   4.5|1147877957|
|     1|   2573|   4.0|1147878923|
|     1|   2632|   5.0|1147878248|
|     1|   2692|   5.0|1147869100|
+------+-------+------+----------+
only showing top 20 rows



In [7]:
# 데이터프레임 스키마 출력
ratings_df.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)



In [20]:
# [+] 타임스탬프를 제외한 컬럼 선택
ratings_df = ratings_df.select(['userID','movieID','rating'])

In [9]:
# [+] describe(): 기술 통계량 출력
ratings_df.select('rating').describe().show()

+-------+------------------+
|summary|            rating|
+-------+------------------+
|  count|             71921|
|   mean|3.5821387355570695|
| stddev| 1.042406032579843|
|    min|               0.5|
|    max|               5.0|
+-------+------------------+



### **2. 훈련 데이터 준비 및 영화 추천 모델 학습**

In [10]:
# [+] 훈련 데이터셋과 테스트 데이터셋 분리 (8:2)
train_df, test_df = ratings_df.randomSplit([0.8,0.2])

In [11]:
# 추천 알고리즘(Alternating Least Squares) 불러오기
from  pyspark.ml.recommendation import ALS

#### **ALS 알고리즘 초매개변수**
+ ```maxIter```: 최대 학습 반복 횟수
+ ```regParam```: 정규화 매개변수(범위: 0~1)
+ ```coldStartStrategy```: 데이터가 부족한 신규 유저 및 아이템에 대한 예측 문제(Cold Start)를 처리하는 방식이며 ```drop```값은 해당 데이터를 모델 학습 과정에서 배제

In [22]:
# 추천 알고리즘 초매개변수 설정
als = ALS(
    maxIter=5,
    regParam=0.1,
    userCol='userID',
    itemCol='movieID',
    ratingCol='rating',
    coldStartStrategy='drop'
)

In [23]:
# [+] 모델 학습
model = als.fit(train_df) # 학습

26/11/26 04:52:56 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/11/26 04:52:56 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/11/26 04:52:56 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


In [24]:
# # 메모리 부족으로 인한 오류 발생시, 아래의 코드를 실행
# from pyspark.sql import SparkSession

# MAX_MEMORY = '5g'
# spark = SparkSession.builder.appName('movie-recommendation')\
#     .config('spark.executor.memory', MAX_MEMORY)\
#     .config('spark.driver.memory', MAX_MEMORY)\
#     .getOrCreate()

In [25]:
# [+] 모델 예측
predictions = model.transform(train_df)

In [26]:
# 예측값 출력
predictions.show()

+------+-------+------+----------+
|userID|movieID|rating|prediction|
+------+-------+------+----------+
|   148|     19|   3.0| 2.9805312|
|   148|     32|   4.0|  4.118832|
|   148|     47|   4.0|  4.247308|
|   148|    110|   4.0| 4.1339498|
|   148|    111|   4.0| 4.1629267|
|   148|    260|   3.5| 4.1749296|
|   148|    293|   4.0|  4.188217|
|   148|    296|   5.0| 4.4176197|
|   148|    356|   4.0| 3.9746184|
|   148|    527|   5.0| 4.2848115|
|   148|    541|   4.0| 4.1656923|
|   148|    589|   4.5|  3.992723|
|   148|    608|   3.0|  4.016892|
|   148|    750|   5.0| 4.2306113|
|   148|    858|   4.5|  4.390253|
|   148|    899|   4.0| 4.0668483|
|   148|    903|   4.0| 4.0410438|
|   148|    904|   4.5| 4.1106095|
|   148|    909|   4.0|   3.96901|
|   148|    912|   4.0| 4.2443933|
+------+-------+------+----------+
only showing top 20 rows



In [27]:
# [+] 평점과 예측평점에 대한 통계 출력
predictions.select(['rating','prediction']).describe().show()

+-------+------------------+------------------+
|summary|            rating|        prediction|
+-------+------------------+------------------+
|  count|             57444|             57444|
|   mean|3.5869977717429147|3.5260022714849253|
| stddev|1.0387239549657141|0.7813636774283785|
|    min|               0.5|         0.2002249|
|    max|               5.0|         5.5800133|
+-------+------------------+------------------+



In [28]:
# 모델 성능 평가: RMSE(Root Mean Squared Error)
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    metricName='rmse',
    labelCol='rating',
    predictionCol='prediction'
)

In [29]:
# [+] RMSE 측정
rmse = evaluator.evaluate(predictions)

In [30]:
rmse

0.5453748860908312

### **3. 학습된 모델을 이용한 영화 추천**
+ ```recommendForAllUsers()```: 유저별 아이템 추천
+ ```recommendForAllItems()```: 아이템별 유저 추천
+ ```recommendForUserSubset()```: 특정 유저 그룹에 대한 아이템 추천

In [31]:
# [+] 유저별 아이템을 3개씩 추천
model.recommendForAllUsers(3).show()

[Stage 161:====================================================> (98 + 1) / 100]

+------+--------------------+
|userID|     recommendations|
+------+--------------------+
|     1|[{6286, 5.346385}...|
|     2|[{5540, 5.8367195...|
|     3|[{6286, 5.105721}...|
|     4|[{171011, 5.10412...|
|     5|[{6286, 5.148163}...|
|     6|[{7071, 5.4731636...|
|     7|[{6286, 4.9114676...|
|     8|[{117364, 5.61031...|
|     9|[{1487, 5.182447}...|
|    10|[{8253, 4.871512}...|
|    11|[{99149, 5.010020...|
|    12|[{3881, 4.8826127...|
|    13|[{8253, 5.208887}...|
|    14|[{3881, 5.6637106...|
|    15|[{170355, 5.73783...|
|    16|[{3881, 5.4740605...|
|    17|[{3881, 5.207331}...|
|    18|[{2467, 4.5042987...|
|    19|[{3881, 5.1491427...|
|    20|[{66371, 5.440341...|
+------+--------------------+
only showing top 20 rows



In [32]:
# [+] 아이템별 유저를 3명씩 추천
model.recommendForAllItems(3).show()

[Stage 190:====================================================> (98 + 1) / 100]

+-------+--------------------+
|movieID|     recommendations|
+-------+--------------------+
|      1|[{327, 4.992286},...|
|      2|[{327, 4.6385493}...|
|      3|[{448, 4.9080725}...|
|      4|[{327, 4.492895},...|
|      5|[{448, 5.0892606}...|
|      6|[{87, 5.1310687},...|
|      7|[{22, 5.079873}, ...|
|      8|[{327, 4.693934},...|
|      9|[{474, 4.76277}, ...|
|     10|[{240, 5.0118117}...|
|     11|[{198, 5.060382},...|
|     12|[{198, 4.819108},...|
|     13|[{327, 5.37024}, ...|
|     14|[{87, 4.9782634},...|
|     15|[{9, 4.5426664}, ...|
|     16|[{198, 5.6781397}...|
|     17|[{22, 5.308078}, ...|
|     18|[{210, 5.106218},...|
|     19|[{199, 4.5276794}...|
|     20|[{210, 4.093128},...|
+-------+--------------------+
only showing top 20 rows



**특정 유저 그룹에 대한 아이템 추천**
1. 해당 유저들의 ID 정보를 데이터프레임으로 생성
2. ```recommendForUserSubset()```를 이용한 아이템 추천

In [36]:
# 특정 유저 ID 집합: 2차원 리스트
user_lst = [[1]]

In [37]:
# 스키마 정의
from pyspark.sql.types import StructType, StructField, IntegerType
schema = StructType([
    StructField('userID', IntegerType(), True)
])

In [38]:
# 데이터프레임 생성
users_df = spark.createDataFrame(user_lst, schema)

In [39]:
users_df.show()

[Stage 206:>                                                        (0 + 1) / 1]

+------+
|userID|
+------+
|     1|
+------+



In [40]:
# [+] recommendForUserSubset(): 특정 유저 그룹에 대한 아이템 추천
user_recs = model.recommendForUserSubset(users_df, 5)

In [41]:
user_recs.show()

+------+--------------------+
|userID|     recommendations|
+------+--------------------+
|     1|[{6286, 5.346385}...|
+------+--------------------+



In [42]:
# 추천 결과를 Python 객체로 받아오기
movies_lst = user_recs.collect()[0].recommendations

In [43]:
movies_lst

[Row(movieID=6286, rating=5.3463850021362305),
 Row(movieID=3881, rating=5.34450101852417),
 Row(movieID=53956, rating=5.174641132354736),
 Row(movieID=8327, rating=4.983951091766357),
 Row(movieID=1484, rating=4.93472957611084)]

In [44]:
# movies_lst에 대한 데이터프레임 생성
recs_df = spark.createDataFrame(movies_lst)
recs_df.show()

+-------+------------------+
|movieID|            rating|
+-------+------------------+
|   6286|5.3463850021362305|
|   3881|  5.34450101852417|
|  53956| 5.174641132354736|
|   8327| 4.983951091766357|
|   1484|  4.93472957611084|
+-------+------------------+



In [45]:
# [+] 영화 데이터에 대한 데이터프레임 생성

file = 'movies.csv'

movies_df = spark.read.csv(path+file, inferSchema=True, header=True)
movies_df.show()

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
|      6|         Heat (1995)|Action|Crime|Thri...|
|      7|      Sabrina (1995)|      Comedy|Romance|
|      8| Tom and Huck (1995)|  Adventure|Children|
|      9| Sudden Death (1995)|              Action|
|     10|    GoldenEye (1995)|Action|Adventure|...|
|     11|American Presiden...|Comedy|Drama|Romance|
|     12|Dracula: Dead and...|       Comedy|Horror|
|     13|        Balto (1995)|Adventure|Animati...|
|     14|        Nixon (1995)|               Drama|
|     15|Cutthroat Island ...|Action|Adventure|...|
|     16|       Casino (1995)|         Crime|Drama|
|     17|Sen

In [48]:
# [+] recs_df, movies_df 에 대한 Temporary View 생성
recs_df.createOrReplaceTempView('recommendations')
movies_df.createOrReplaceTempView('movies')

In [49]:
# SQL JOIN 연산을 통해 추천된 영화 제목 받아오기
spark.sql("""
    SELECT *
    FROM movies JOIN recommendations
    ON movies.movieID = recommendations.movieID
    ORDER BY rating DESC
""").show()

+-------+--------------------+--------------------+-------+------------------+
|movieId|               title|              genres|movieID|            rating|
+-------+--------------------+--------------------+-------+------------------+
|   6286|Man Without a Pas...|  Comedy|Crime|Drama|   6286|5.3463850021362305|
|   3881|Phish: Bitterswee...|         Documentary|   3881|  5.34450101852417|
|  53956|Death at a Funera...|              Comedy|  53956| 5.174641132354736|
|   8327|        Dolls (2002)|       Drama|Romance|   8327| 4.983951091766357|
|   1484|Daytrippers, The ...|Comedy|Drama|Myst...|   1484|  4.93472957611084|
+-------+--------------------+--------------------+-------+------------------+



### **4. 유저 별 영화 추천 서비스를 간단하게 구현하기**
1. SQL문 작성
2. 영화 추천 함수 작성
3. 영화 추천 테스트

In [ ]:
# SQL JOIN 연산을 통해 추천된 영화 제목 받아오기
query = """
SELECT * 
FROM movies JOIN recommendations 
ON movies.movieID = recommendations.movieID
ORDER BY rating DESC
"""

In [ ]:
# 입력된 유저에 대한 영화 추천 함수
def get_recommendations(user_id, num_recs):
    users_df = spark.createDataFrame([user_id], IntegerType()).toDF('userID')
    users_recs_df = model.recommendForUserSubset(users_df, num_recs)
    recs_lst = users_recs_df.collect()[0].recommendations
    recs_df = spark.createDataFrame(recs_lst)
    recs_df.createOrReplaceTempView('recommendations')
    recommended_movies = spark.sql(query)
    
    return recommended_movies

In [ ]:
# 1번 유저에 대한 영화 5개 추천
recs = get_recommendations(1, 5)

In [ ]:
# 추천 결과 출력
recs.show()

In [ ]:
# toPandas(): Pandas 데이터프레임으로 출력
recs.toPandas()